In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

DATA_DIR_CANDIDATES = (Path("deepgram"), Path("."))
DATA_DIR = next(
    (path for path in DATA_DIR_CANDIDATES if (path / "deepgram_dataset_with_features.csv").is_file()),
    None,
)
SPLIT_DIR_CANDIDATES = (Path("datasets/deepgram"), Path("../datasets/deepgram"), DATA_DIR)
SPLIT_DIR = next(
    (path for path in SPLIT_DIR_CANDIDATES if (path / "training_422.csv").is_file()),
    None,
)
if DATA_DIR is None or SPLIT_DIR is None:
    raise FileNotFoundError("Could not locate the DeepGram data files.")

combined_dataset = pd.read_csv(DATA_DIR / "deepgram_dataset_with_features.csv")


## Standardization and statistical feature selection

In [ ]:
deepgram_features_to_scale = combined_dataset.drop(columns=["data", "label"])
scaled_features = StandardScaler().fit_transform(deepgram_features_to_scale)
scaled_df = pd.DataFrame(scaled_features, columns=deepgram_features_to_scale.columns)
scaled_df["data"] = combined_dataset["data"].values
scaled_df["label"] = combined_dataset["label"].values
combined_dataset = scaled_df[["data", "label", *deepgram_features_to_scale.columns]]

group_0 = combined_dataset.loc[combined_dataset["label"] == 0].drop(columns=["data", "label"])
group_1 = combined_dataset.loc[combined_dataset["label"] == 1].drop(columns=["data", "label"])
t_stats, p_values = ttest_ind(group_0, group_1, equal_var=False)
ttest_results = pd.DataFrame(
    {"feature": group_0.columns, "t_stat": t_stats, "p_value": p_values}
)
nan_features = ttest_results.loc[ttest_results["p_value"].isna(), "feature"]
combined_dataset = combined_dataset.drop(
    columns=[column for column in nan_features if column in combined_dataset.columns]
)
ttest_results = ttest_results.dropna()
p_values_df = ttest_results[["feature", "p_value"]].copy()
_, q_values, _, _ = multipletests(p_values_df["p_value"], method="fdr_bh")
p_values_df["q_value"] = q_values
significant_features = p_values_df.loc[p_values_df["q_value"] < 0.01, "feature"].tolist()
cleaned_data = combined_dataset[["data", "label", *significant_features]]


## Feature-group selection

In [ ]:
selected_feature_files = (
    "deepgram_features_aac.csv",
    "deepgram_features_dipeptide.csv",
    "deepgram_features_physico_chemical_transition.csv",
    "deepgram_features_normalized_moreau_broto_autocor.csv",
    "deepgram_features_quasi.csv",
    "deepgram_features_soc.csv",
    "deepgram_features_pseudo_amino_acid_composition.csv",
)
selected_feature_columns = {
    column
    for filename in selected_feature_files
    for column in pd.read_csv(DATA_DIR / filename).columns
}
selected_columns = [
    column for column in cleaned_data.columns
    if column in {"data", "label", *selected_feature_columns}
]
new_cleaned_data = cleaned_data[selected_columns]


## Train-validation split preparation

In [ ]:
deepgram_train = pd.concat(
    [
        pd.read_csv(SPLIT_DIR / "training_422.csv"),
        pd.read_csv(SPLIT_DIR / "trainingNon_422.csv"),
    ],
    ignore_index=True,
).sample(frac=1).reset_index(drop=True)
deepgram_validation = pd.concat(
    [
        pd.read_csv(SPLIT_DIR / "independent_150.csv"),
        pd.read_csv(SPLIT_DIR / "independentNon_150.csv"),
    ],
    ignore_index=True,
).sample(frac=1).reset_index(drop=True)

deepgram_train = deepgram_train.drop_duplicates(subset="data", keep=False).reset_index(drop=True)
deepgram_validation = deepgram_validation.drop_duplicates(subset="data", keep=False).reset_index(drop=True)
common_peptides = set(deepgram_train["data"]).intersection(deepgram_validation["data"])
train_common = deepgram_train.loc[
    deepgram_train["data"].isin(common_peptides), ["data", "label"]
].rename(columns={"label": "label_train"})
validation_common = deepgram_validation.loc[
    deepgram_validation["data"].isin(common_peptides), ["data", "label"]
].rename(columns={"label": "label_validation"})
result = pd.merge(train_common, validation_common, on="data")
conflicting_peptides = result.loc[
    result["label_train"] != result["label_validation"], "data"
].tolist()
deepgram_train = deepgram_train[
    ~deepgram_train["data"].isin(conflicting_peptides)
].reset_index(drop=True)
deepgram_validation = deepgram_validation[
    ~deepgram_validation["data"].isin(conflicting_peptides)
].reset_index(drop=True)
result = result[~result["data"].isin(conflicting_peptides)].reset_index(drop=True)

label_zero = result.loc[result["label_train"] == 0, "data"].tolist()
label_one = result.loc[result["label_train"] == 1, "data"].tolist()
train_holdout = label_zero[:56] + label_one[: len(label_one) // 2]
validation_holdout = label_zero[56:] + label_one[len(label_one) // 2 :]
deepgram_train = deepgram_train[
    ~deepgram_train["data"].isin(train_holdout)
].reset_index(drop=True)
deepgram_validation = deepgram_validation[
    ~deepgram_validation["data"].isin(validation_holdout)
].reset_index(drop=True)

deepgram_train_df = new_cleaned_data[
    new_cleaned_data["data"].isin(deepgram_train["data"])
].copy()
deepgram_validation_df = new_cleaned_data[
    new_cleaned_data["data"].isin(deepgram_validation["data"])
].copy()


## Random Forest model

In [ ]:
X_train = deepgram_train_df.drop(columns=["data", "label"])
y_train = deepgram_train_df["label"]
X_validation = deepgram_validation_df.drop(columns=["data", "label"])
y_validation = deepgram_validation_df["label"]

def specificity_score(y_true, y_pred):
    tn, fp, _, _ = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "mcc": make_scorer(matthews_corrcoef),
    "specificity": make_scorer(specificity_score),
}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1,
)

def mean_std(scores):
    return f"{np.mean(scores):.4f} +/- {np.std(scores):.4f}"

print(f"Train size: {len(X_train)}, Validation size: {len(X_validation)}")
print("\n10-Fold CV Results (Train Set - Random Forest):")
for metric, label in (
    ("accuracy", "Accuracy"),
    ("f1", "F1 Score"),
    ("recall", "Sensitivity"),
    ("specificity", "Specificity"),
    ("precision", "Precision"),
    ("roc_auc", "AUC (ROC)"),
    ("mcc", "MCC"),
):
    print(f"{label:<13}: {mean_std(cv_results[f'test_{metric}'])}")

model.fit(X_train, y_train)
y_pred = model.predict(X_validation)
y_probability = model.predict_proba(X_validation)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_validation, y_pred).ravel()
print("\nValidation Set Results (Random Forest):")
print(f"Accuracy     : {accuracy_score(y_validation, y_pred):.4f}")
print(f"F1 Score     : {f1_score(y_validation, y_pred):.4f}")
print(f"Sensitivity  : {recall_score(y_validation, y_pred):.4f}")
print(f"Specificity  : {tn / (tn + fp):.4f}")
print(f"Precision    : {precision_score(y_validation, y_pred):.4f}")
print(f"AUC (ROC)    : {roc_auc_score(y_validation, y_probability):.4f}")
print(f"MCC          : {matthews_corrcoef(y_validation, y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_validation, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_validation, y_pred)}")
